# 01 Dataset Preparation

In [1]:
!git clone https://github.com/JeserylMae/unet-ensemble.git

Cloning into 'unet-ensemble'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 41 (delta 13), reused 36 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 31.56 KiB | 10.52 MiB/s, done.
Resolving deltas: 100% (13/13), done.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys
sys.path.append('/content/unet-ensemble')

## 01-1 Download MidV-500

In [ ]:
!pip install midv500

In [ ]:
import midv500  

### Download and unzip desired version of the dataset:

In [ ]:
# set directory for dataset to be downloaded
dataset_dir = 'midv500_data/'

In [ ]:
# download and unzip the base midv500 dataset
dataset_name = "midv500"
midv500.download_dataset(dataset_dir, dataset_name)

### Convert downloaded dataset to coco format:

In [ ]:
# set directory for coco annotations to be saved
export_dir = 'midv500_data_anno/'

# set the desired name of the coco file, coco file will be exported as "filename + '_coco.json'"
filename = 'midv500'

# convert midv500 annotations to coco format
midv500.convert_to_coco(dataset_dir, export_dir, filename)

### Copy dataset to Google Drive

Copy all

In [ ]:
!cp -r /content/midv500_data/ /content/drive/MyDrive/MidV-500/

Copy all except video folders

In [ ]:
!rsync -av --progress /content/midv500_data/midv500/ /content/drive/MyDrive/MidV-500/ --exclude 'videos/'

Copy only the first tif file

In [ ]:
import os
import shutil

src_base = '/content/midv500_data/midv500'
dst_base = '/content/drive/MyDrive/MidV-500-first-tif'

os.makedirs(dst_base, exist_ok=True)

# Loop through all country folders
for country_folder in os.listdir(src_base):
    country_path = os.path.join(src_base, country_folder, 'images')
    print(country_folder)
    if os.path.exists(country_path):
        # Find all .tif files
        tif_files = [f for f in os.listdir(country_path) if f.lower().endswith('.tif')]
        if tif_files:
            first_tif = tif_files[0]  # get the first tif file
            src_file = os.path.join(country_path, first_tif)
            dst_file = os.path.join(dst_base, f"{country_folder}_{first_tif}")
            shutil.copy(src_file, dst_file)
            print(f"Copied {src_file} -> {dst_file}")


## 01-2 Authentic Images Generation Using DocXPand Generator

In [ ]:
# !git clone https://github.com/QuickSign/docxpand.git
!git clone https://github.com/JeserylMae/docxpand.git

%cd docxpand

In [ ]:
!pip install poetry
!poetry install

In [ ]:
!poetry run python scripts/dataset/generate_fake_structured_documents.py -n 565 -o template -t 'midv_chile' -s https://mangily-noneidetic-jonie.ngrok-free.dev

In [ ]:
# Copy all files from template-b that have 'front.svg' in the name
!cp /content/docxpand/template-c/*front.svg* "output_dir"

## 01-3 Image Format Conversion 

In [ ]:
!apt-get update
!apt-get install -y librsvg2-bin

In [ ]:
import os
import subprocess
from PIL import Image

input_folder = "/content/drive/MyDrive/template-c"
output_folder = "/content/drive/MyDrive/template-c-jpg"

counter = 0

target_width = 856
target_height = 540

dpi = 1200

jpeg_quality = 95
jpeg_subsampling = 0

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if not filename.lower().endswith(".svg"):
        continue

    svg_path = os.path.join(input_folder, filename)
    png_path = os.path.join(output_folder, os.path.splitext(filename)[0] + ".png")
    jpg_path = os.path.join(output_folder, os.path.splitext(filename)[0] + ".jpg")

    subprocess.run([
        "rsvg-convert",
        "--format=png",
        "--background-color=white",
        f"--dpi-x={dpi}",
        f"--dpi-y={dpi}",
        f"--width={target_width}",
        f"--height={target_height}",
        svg_path,
        "--output", png_path
    ], check=True)

    with Image.open(png_path) as img:
        img.convert("RGB").save(
            jpg_path,
            "JPEG",
            quality=jpeg_quality,
            subsampling=jpeg_subsampling
        )

    os.remove(png_path)

    counter += 1
    print(f"Converted {counter}: {filename} → JPG")

print("🎉 All SVGs converted successfully!")


## 01-4 Dataset Normalization

In [4]:
!pip install opencv-python matplotlib tqdm

In [5]:
import os
import cv2
import numpy as np

from tqdm import tqdm
from src.utils.normalize import Normalize

In [12]:
INPUT_DIR = "/content/drive/Shareddrives/U-Net Ensemble/Dataset/Raw/Evaluation/Synthetic-1/temp-a"
OUTPUT_DIR = "/content/drive/Shareddrives/U-Net Ensemble/Dataset/Raw/Evaluation/Synthetic-1/temp-a-norm"

os.makedirs(OUTPUT_DIR, exist_ok=True)

### Normalization Only

In [13]:
image_extensions = [".jpg", ".jpeg", ".png"]

for filename in tqdm(os.listdir(INPUT_DIR)):
    if any(filename.lower().endswith(ext) for ext in image_extensions):
        input_path = os.path.join(INPUT_DIR, filename)
        output_path = os.path.join(OUTPUT_DIR, filename)

        img = cv2.imread(input_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        height, width = img.shape[:2]

        scale = 256 / max(width, height)
        new_w = int(width * scale)
        new_h = int(height * scale)

        final = Normalize().normalize(img, new_w, new_h, resize=True)

        cv2.imwrite(output_path, final)

print("Preprocessing complete!")

100%|██████████| 67/67 [00:04<00:00, 16.08it/s]

Preprocessing complete!


### Normalization w/ Cropping

In [ ]:
image_extensions = [".jpg", ".jpeg", ".png"]

for filename in tqdm(os.listdir(INPUT_DIR)):
    if any(filename.lower().endswith(ext) for ext in image_extensions):
        input_path = os.path.join(INPUT_DIR, filename)
        output_path = os.path.join(OUTPUT_DIR, filename)

        # if "template-a" in filename:
        # edges, img = get_edges_template_a(input_path)
        # else:
        edges, img = Normalize().get_edges(input_path)

        warped, width, height = Normalize().crop_image(edges, img)
        final = Normalize().normalize(warped, width, height)

        cv2.imwrite(output_path, final)

print("Preprocessing complete!")

In [ ]:
!cp -r "/content/Normalized/template-a/" "/content/drive/Shareddrives/U-Net Ensemble/Dataset/temp"